# VADAR Video Analysis Quickstart

This notebook demonstrates how to use VADAR's extended video analysis capabilities, including:
1. **Video Loading & Frame Extraction**
2. **Object Tracking with SAM 2**
3. **End-to-End Video Question Answering**

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path if needed
sys.path.append('..')

## 1. Setup Models
Initialize the `VideoModulesList`. This will load SAM 2 in video mode, GroundingDINO, and Molmo/GPT-4o.

In [ ]:
import torch
from engine.predefined_modules import VideoModulesList

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize video-aware modules
modules = VideoModulesList(device=device)
print("Modules loaded successfully.")

## 2. Video Question Answering
Use the `VideoEngine` to answer a question about a video file.

In [ ]:
from engine.video_engine import VideoEngine

# Path to your sample video
video_path = "path/to/your/video.mp4"
question = "How many people appear in the video?"

if os.path.exists(video_path):
    engine = VideoEngine(modules)
    result = engine.run(video_path, question)
    
    print(f"Question: {question}")
    print(f"Answer: {result}")
else:
    print(f"Please provide a valid video path at: {video_path}")

## 3. Manual Tracking Breakdown
You can also use the `track` module directly to see how SAM 2 propagates masks.

In [ ]:
from engine.video_utils import extract_frames

video_dir = "./temp_frames"
if os.path.exists(video_path):
    # Extract frames
    extract_frames(video_path, video_dir, fps=2)
    
    # Track an object starting from a point in frame 0
    # (x, y) coordinates of the object to track
    initial_point = (100, 200) 
    
    bboxes = modules.track(video_dir, init_frame_index=0, x=initial_point[0], y=initial_point[1])
    
    print(f"Tracked across {len(bboxes)} frames.")
    print("Sample bounding boxes:")
    for i in range(min(5, len(bboxes))):
        print(f"Frame {i}: {bboxes[i]}")